# GradeScope — 06. Eksport modeli do ONNX

In [1]:
import pickle
import json
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

with open("../data/splits.pkl", "rb") as f:
    data = pickle.load(f)

with open("../data/selected_features.pkl", "rb") as f:
    selected_features = pickle.load(f)

with open("../data/trained_pipelines.pkl", "rb") as f:
    pipelines = pickle.load(f)

with open("../data/anomaly_detector.pkl", "rb") as f:
    anomaly_detector = pickle.load(f)

X_train = data["X_train"][selected_features]
X_test  = data["X_test"][selected_features]
y_test  = data["y_test"]
n_features = len(selected_features)

print(f"Modele: {list(pipelines.keys())}")
print(f"Cechy: {n_features} — {selected_features}")

Modele: ['random_forest', 'svm', 'knn', 'neural_network']
Cechy: 12 — ['Attendance', 'Hours_Studied', 'Previous_Scores', 'Tutoring_Sessions', 'Sleep_Hours', 'Parental_Involvement', 'Access_to_Resources', 'Physical_Activity', 'Peer_Influence', 'Family_Income', 'Parental_Education_Level', 'Distance_from_Home']


## 1. Eksport Pipeline'ów do ONNX

In [2]:
initial_type = [("float_input", FloatTensorType([None, n_features]))]
target_opset = {"ai.onnx.ml": 3, "": 17}

# zipmap=False — probability output as float32 tensor (required for onnxruntime-web)
for name, pipeline in pipelines.items():
    onnx_model = convert_sklearn(
        pipeline,
        initial_types=initial_type,
        options={"zipmap": False},
        target_opset=target_opset,
    )
    path = f"../models/{name}.onnx"
    with open(path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    size_kb = len(onnx_model.SerializeToString()) / 1024
    print(f"✓ {name}.onnx — {size_kb:.1f} KB")

✓ random_forest.onnx — 21993.6 KB
✓ svm.onnx — 432.3 KB


✓ knn.onnx — 1026.4 KB
✓ neural_network.onnx — 67.6 KB


## 2. Weryfikacja — ONNX vs sklearn

In [3]:
# Porównanie predykcji sklearn vs ONNX na zbiorze testowym
X_test_f32 = X_test.values.astype(np.float32)

for name in pipelines:
    sess = rt.InferenceSession(f"../models/{name}.onnx")
    input_name = sess.get_inputs()[0].name

    pred_onnx    = sess.run(None, {input_name: X_test_f32})[0]
    pred_sklearn = pipelines[name].predict(X_test)

    zgodnosc = np.mean(pred_onnx == pred_sklearn) * 100
    print(f"{name}: zgodność ONNX vs sklearn = {zgodnosc:.2f}%")

random_forest: zgodność ONNX vs sklearn = 99.77%


svm: zgodność ONNX vs sklearn = 73.22%


knn: zgodność ONNX vs sklearn = 100.00%
neural_network: zgodność ONNX vs sklearn = 100.00%


## 2b. Eksport detektora anomalii

In [4]:
# IsolationForest nie wymaga skalowania (model drzewowy), eksport bez wrappera Pipeline
onnx_anomaly = convert_sklearn(
    anomaly_detector,
    initial_types=[("float_input", FloatTensorType([None, n_features]))],
    target_opset=target_opset,
)
anomaly_path = "../models/anomaly.onnx"
with open(anomaly_path, "wb") as f:
    f.write(onnx_anomaly.SerializeToString())
size_kb = len(onnx_anomaly.SerializeToString()) / 1024
print(f"✓ anomaly.onnx — {size_kb:.1f} KB")

# Weryfikacja
sess = rt.InferenceSession(anomaly_path)
input_name = sess.get_inputs()[0].name
X_test_f32 = X_test.values.astype(np.float32)
pred_onnx    = sess.run(None, {input_name: X_test_f32})[0]
pred_sklearn = anomaly_detector.predict(X_test)
zgodnosc = np.mean(pred_onnx == pred_sklearn) * 100
print(f"anomaly: zgodnosc ONNX vs sklearn = {zgodnosc:.2f}%")

✓ anomaly.onnx — 1175.5 KB


anomaly: zgodnosc ONNX vs sklearn = 88.76%


## 3. Zapis metadanych dla frontendu

In [5]:
df_train = data["X_train"][selected_features]

feature_meta = {}
for col in selected_features:
    feature_meta[col] = {
        "min":  int(df_train[col].min()),
        "max":  int(df_train[col].max()),
        "mean": round(float(df_train[col].mean()), 2),
    }

metadata = {
    "features":     selected_features,
    "feature_meta": feature_meta,
    "models":        list(pipelines.keys()),
    "pass_threshold": 67,
}

with open("../data/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Zapisano: model_metadata.json")
print("Modele:", metadata["models"])

Zapisano: model_metadata.json
Modele: ['random_forest', 'svm', 'knn', 'neural_network']


## 4. Kopiowanie plików do frontendu

In [6]:
import shutil, os

src_dir = "../models"
dst_dir = "../frontend/public/models"
os.makedirs(dst_dir, exist_ok=True)

for fname in os.listdir(src_dir):
    if fname.endswith(".onnx"):
        shutil.copy(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
        print(f"Skopiowano: {fname}")

shutil.copy("../data/model_metadata.json", "../frontend/public/model_metadata.json")
print("Skopiowano: model_metadata.json")

Skopiowano: anomaly.onnx
Skopiowano: gradient_boosting.onnx
Skopiowano: knn.onnx
Skopiowano: neural_network.onnx


Skopiowano: random_forest.onnx
Skopiowano: svm.onnx


Skopiowano: model_metadata.json
